In [1]:
import pandas  as pd
import numpy as np
import pdb, os, datetime, itertools, time, hashlib
from dotenv import load_dotenv

load_dotenv()
from kdutils.macro2 import base_path
from lumina.genetic.util import create_id

In [2]:
method = 'cicso0'
task_id = '1000201201'

In [3]:
def create_params(params):
    m = hashlib.md5()
    # params可能是字典类型，需要转换为字符串
    if isinstance(params, dict):
        # 将字典按键排序后转换为字符串，确保相同参数组合产生相同hash
        params_str = str(sorted(params.items()))
    else:
        params_str = str(params)
    m.update(bytes(params_str, encoding='UTF-8'))
    return create_id(original=m.hexdigest(), digit=16)

In [4]:
def screening(method, task_id):
    output_dirs = os.path.join(base_path, method, 'evaluate', str(task_id))
    basic_csv = pd.read_csv(os.path.join(output_dirs, "basic", "summary.csv"),
                            index_col=0)
    basic_csv['category'] = 'basic'
    
    derivative_csv = pd.read_csv(os.path.join(output_dirs, "derivative",
                                              "summary.csv"),
                                 index_col=0)
    derivative_csv['category'] = 'derivative'
    results = pd.concat([basic_csv, derivative_csv], axis=0)
    results['abs_ic'] = np.fabs(results['ic'])
    results = results.sort_values(by=['abs_ic'],ascending=False).dropna()
    results = results[(results['abs_ic'] > 0.1) & (results['abs_ic'] < 0.5) & (results['turnover'] < 0.7)]
    results['factor_id'] = results['name'].apply(lambda x: create_params(x))
    return results

In [5]:
results = screening(method, task_id)

In [6]:
results = results[['name','factor_id','category','abs_ic','icir','turnover']]
results.head()

,name,factor_id,category,abs_ic,icir,turnover
4,LAST('ak001_1_15_0'),1072123776613351,basic,0.191342,2.136675,0.681272
319,"MRes(20, MCORR(10, 'ak101_1_3_1', MSUM(10, 'ak...",1071953647919478,derivative,0.182732,2.071571,0.680736
6,LAST('ak002_1_2_5_1'),1083768262857260,basic,0.178293,1.860466,0.628928
302,"MRes(16, EMA(10, MCORR(10, 'ak023_1_3_1', 'ak1...",1027676099869173,derivative,0.176879,2.015556,0.676800
503,"MUL(MSTD(18, MUL('ak106_1_15_0', 'ak104_1_3_1'...",1087933007054047,derivative,0.176833,2.024085,0.637467


In [7]:
#results.apply(lambda x: os.path.join(base_path, 'evaluate', str(task_id), x['category'], 'plot', f"{x['factor_id']}.png"))

results['path'] = (base_path + '/' + str(method) + '/' + 'evaluate' + '/' + str(task_id) + '/' + results['category'].astype(str) \
    + '/' + 'plot' + '/' + results['factor_id'].astype(str) + '.png')
results.head()

,name,factor_id,category,abs_ic,icir,turnover,path
4,LAST('ak001_1_15_0'),1072123776613351,basic,0.191342,2.136675,0.681272,./records/cicso0/evaluate/1000201201/basic/plo...
319,"MRes(20, MCORR(10, 'ak101_1_3_1', MSUM(10, 'ak...",1071953647919478,derivative,0.182732,2.071571,0.680736,./records/cicso0/evaluate/1000201201/derivativ...
6,LAST('ak002_1_2_5_1'),1083768262857260,basic,0.178293,1.860466,0.628928,./records/cicso0/evaluate/1000201201/basic/plo...
302,"MRes(16, EMA(10, MCORR(10, 'ak023_1_3_1', 'ak1...",1027676099869173,derivative,0.176879,2.015556,0.676800,./records/cicso0/evaluate/1000201201/derivativ...
503,"MUL(MSTD(18, MUL('ak106_1_15_0', 'ak104_1_3_1'...",1087933007054047,derivative,0.176833,2.024085,0.637467,./records/cicso0/evaluate/1000201201/derivativ...


In [10]:
results = results.drop(['factor_id'],axis=1)
results.head()

,name,category,abs_ic,icir,turnover,path
4,LAST('ak001_1_15_0'),basic,0.191342,2.136675,0.681272,./records/cicso0/evaluate/1000201201/basic/plo...
319,"MRes(20, MCORR(10, 'ak101_1_3_1', MSUM(10, 'ak...",derivative,0.182732,2.071571,0.680736,./records/cicso0/evaluate/1000201201/derivativ...
6,LAST('ak002_1_2_5_1'),basic,0.178293,1.860466,0.628928,./records/cicso0/evaluate/1000201201/basic/plo...
302,"MRes(16, EMA(10, MCORR(10, 'ak023_1_3_1', 'ak1...",derivative,0.176879,2.015556,0.676800,./records/cicso0/evaluate/1000201201/derivativ...
503,"MUL(MSTD(18, MUL('ak106_1_15_0', 'ak104_1_3_1'...",derivative,0.176833,2.024085,0.637467,./records/cicso0/evaluate/1000201201/derivativ...


In [11]:
def make_clickable(val):
    return f'<a target="_blank" href="{val}">{val}</a>'

In [12]:
results['path'] = results['path'].apply(make_clickable)

In [13]:
from IPython.display import display, HTML

In [14]:
display(HTML(results.to_html(escape=False)))

,name,category,abs_ic,icir,turnover,path
4,LAST('ak001_1_15_0'),basic,0.191342,2.136675,0.681272,./records/cicso0/evaluate/1000201201/basic/plot/1072123776613351.png
319,"MRes(20, MCORR(10, 'ak101_1_3_1', MSUM(10, 'ak024_1_15_0')), 'ak001_1_15_0')",derivative,0.182732,2.071571,0.680736,./records/cicso0/evaluate/1000201201/derivative/plot/1071953647919478.png
6,LAST('ak002_1_2_5_1'),basic,0.178293,1.860466,0.628928,./records/cicso0/evaluate/1000201201/basic/plot/1083768262857260.png
302,"MRes(16, EMA(10, MCORR(10, 'ak023_1_3_1', 'ak103_1_3_1')), 'ak001_1_15_0')",derivative,0.176879,2.015556,0.676800,./records/cicso0/evaluate/1000201201/derivative/plot/1027676099869173.png
503,"MUL(MSTD(18, MUL('ak106_1_15_0', 'ak104_1_3_1')), 'ak001_1_15_0')",derivative,0.176833,2.024085,0.637467,./records/cicso0/evaluate/1000201201/derivative/plot/1087933007054047.png
8,LAST('ak002_1_3_10_1'),basic,0.172959,1.833256,0.529266,./records/cicso0/evaluate/1000201201/basic/plot/1015153864860072.png
306,"MRes(18, MUL('ak013_1_3_1', RSI(18, 'ak105_1_3_1')), 'ak002_1_2_5_1')",derivative,0.171522,1.804004,0.640048,./records/cicso0/evaluate/1000201201/derivative/plot/1062477320377172.png
152,LAST('ak108_1_15_0'),basic,0.167497,2.013734,0.588441,./records/cicso0/evaluate/1000201201/basic/plot/1094504582391988.png
510,"MUL(RSI(4, MRes(12, 'ak102_1_3_0', 'ak102_1_3_0')), 'ak108_1_15_0')",derivative,0.167478,2.013526,0.588434,./records/cicso0/evaluate/1000201201/derivative/plot/1034243530180748.png
2,"ADDED('ak108_1_15_0', MRes(10, 'ak006_1_5_1', MSTD(12, 'ak011_1_3_0')))",derivative,0.167178,1.990643,0.588858,./records/cicso0/evaluate/1000201201/derivative/plot/1027057706704962.png
